# Youtube Data API: Respuestas Comentarios
https://developers.google.com/youtube/v3/docs/comments

https://developers.google.com/youtube/v3/docs/commentThreads

## Contenido:

* [Librerias](*Librerias)
* [Videos](*Videos)


<a name="Librerias"></a>
## Librerias

In [ ]:
# ========= Data Cleaning =========

!pip install advertools
!pip install clean-text
!pip install urlextract
import advertools as adv
from urlextract import URLExtract
from nltk.tokenize import regexp_tokenize
import html
import string 
import re
from cleantext import clean
import unicodedata

# ========= Data Cleaning =========

import pandas as pd
import requests
import csv
import time
from datetime import datetime
from datetime import date
import os.path
from os import path
import math
from google.colab import drive
key='TU_CLAVE_DE_API'
drive.mount('/content/drive/')

#### https://www.youtube.com/playlist?list=
#### https://www.youtube.com/channel/
#### https://www.youtube.com/watch?v=

## Respuestas de Comentarios

### Ejecucion

#### Parametros constantes

In [ ]:
URIcr='https://www.googleapis.com/youtube/v3/comments?key='

# ======parametros requeridos=====

part='snippet,id' #parametro obligatorio

# =====parametros opcionales=====

maxResults='1' #numero maximo de elementos a devolver. El rango es 1 a 100, no se puede usar con el parametro id.
textFormat='html' #comentarios con formato HTML o como texto sin formato. , html= comentarios con formato HTML (valor predeterminado), 
# plainText = comentarios en formato de texto sin formato

#### Parametros variables

In [ ]:
#=====filtros (especifique exactamente uno de los siguientes parámetros)=====

#allThreadsRelatedToChannelId = devuelve todos los hilos de comentarios asociados con el canal especificado. 
# La respuesta puede incluir comentarios sobre el canal o sobre los videos del canal.
#channelId= devuelve hilos de comentarios que contengan comentarios sobre el canal especificado. 
# (La respuesta no incluirá los comentarios dejados en los videos que subió el canal).
#id= lista separada por comas de ids de comentarios

parentId ='UgzptWDhwsVRwYQEP8p4AaABAg' #devuelve hilos de comentarios asociados con el ID de video especificado

# =====parametros opcionales=====

pageToken='' #pagina especifica que se debe devolver, no se puede usar con el parametro id

#### Solicitud

In [ ]:
r = requests.get(URIcr+key+'&part='+part+'&parentId='+parentId\
                 +'&maxResults='+maxResults+'&pageToken='+pageToken+'&textFormat='+textFormat)

In [ ]:
r

In [ ]:
r.reason

In [ ]:
d = dict(r.json())

In [ ]:
d.keys()

In [ ]:
d

#### Resultados

In [ ]:
items = d.get('items')
items

#### Dataframe

In [ ]:
data=pd.json_normalize(items)
data.info()

In [ ]:
data

### Algoritmo en Produccion

#### Importando lista de Comentarios 

In [ ]:
# ================== OJO modificar para cada serie ==================

Comentarios = pd.read_csv ('/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/YoutubeResults/Comentarios/\
Narcos/2022-09-13/RESULTS-COMENTARIOS-Fecha2022-09-13 17:09:18.158791.csv')

# ================== OJO modificar para cada serie ==================

Comentarios

In [ ]:
Comentarios.info()

In [ ]:
Comentarios=Comentarios[Comentarios.ReplicasTotalesComentarioPrincipal.notnull()]
Comentarios.info()

In [ ]:
Comentarios[['ReplicasTotalesComentarioPrincipal']] = Comentarios[['ReplicasTotalesComentarioPrincipal']].astype(int)
Comentarios.info()

In [ ]:
Comentarios_mask=Comentarios['ReplicasTotalesComentarioPrincipal']!=0
filtered_Comentarios = Comentarios[Comentarios_mask]
filtered_Comentarios.info()

In [ ]:
dataCM=filtered_Comentarios.loc[:,['idVideo','idComentarioPrincipal','TextoComentarioPrincipal','ReplicasTotalesComentarioPrincipal']]
dataCM

In [ ]:
dataCM['ReplicasTotalesComentarioPrincipal'].sum()

In [ ]:
ListaIDs=list(dataCM['idComentarioPrincipal'])
len(ListaIDs)

#### Parametros constantes

In [ ]:
URIcr = 'https://www.googleapis.com/youtube/v3/comments?key='

# ================== Parametros Requeridos ==================

part = 'snippet,id' #parametro obligatorio
fields = 'pageInfo(*),nextPageToken(*),\
items(id,snippet(textOriginal,parentId,authorDisplayName,authorChannelUrl\
,authorChannelId.value,canRate,viewerRating,likeCount,publishedAt,updatedAt))'

# ================== Parametros Opcionales ==================

maxResults = '100' # Numero maximo de elementos a devolver. El rango es 1 a 100, no se puede usar con el parametro id.
textFormat = 'html' # Comentarios con formato HTML o como texto sin formato. html = comentarios con formato HTML (valor predeterminado), 
# plainText = comentarios en formato de texto sin formato

#### Parametros variables

In [ ]:
# ================== Filtros (especifique exactamente uno de los siguientes parámetros) ==================

ListIdComentarios = ListaIDs # Devuelve hilos de comentarios asociados con el ID de video especificado

# ================== Parametros Opcionales ==================

pageToken = '' # Pagina especifica que se debe devolver, no se puede usar con el parametro id

#### Carpetas Drive

In [ ]:
today = str(date.today())

# ======================= OJO Cambiar PATH para cada Serie =======================

PATH_TO_SAVE_SEARCHES = '/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/YoutubeResults/RespuestasComentarios/Narcos/'

# ======================= OJO Cambiar PATH para cada Serie =======================

PATH_TO_DIR_0 = PATH_TO_SAVE_SEARCHES + today
if os.path.exists(PATH_TO_DIR_0) == False:
  os.mkdir(PATH_TO_DIR_0)
file_nameL = 'RESPUESTASCOMENTARIOS'+str(datetime.now())

#### Constantes

In [ ]:
solicitud = 0 # Inicio de las solicitudes
espera = 0.5 # Valor de retardo para las solicitudes
solicitudmax = 10000 # Numero de solicitudes maximas (el limite es 10000)

# ======================= OJO Cambiar diariamente =======================

ithid=4996 # i-esimo id

# ======================= OJO Cambiar diariamente ======================= 

#### Ejecucion

In [ ]:
Datac = pd.DataFrame() # Dataframe de Resultados Totales

Metadatac = pd.DataFrame(columns=['Solicitud','idComentarioPrincipal','maxResultados','FechaEjecucion','Respuesta'\
                                  ,'prevPageToken','nextPageToken','Resultadosxpagina']) # Dataframe de Informacion de Solicitudes

Faltantes=pd.DataFrame(columns=['idComentarioPrincipal','Solicitud'])

for idcomentario in ListIdComentarios[ithid:]: # For de id's de comentarios

  if (solicitud == solicitudmax+1):
    break

  filtro=dataCM[dataCM['idComentarioPrincipal'] == idcomentario]
  ReplicasTotales=list(filtro['ReplicasTotalesComentarioPrincipal'])[0]
  
  paginasmax = math.ceil(ReplicasTotales/100) # Numero de paginas maxinas para los comentarios
  pageToken = '' # Pagina especifica que se debe devolver OJO DEBE DE IR EN ESTE LUGAR

  for pagina in range(1,paginasmax+1): # For de la paginacion

    solicitud = solicitud+1
    
    if (solicitud == solicitudmax+1):
      print('Comentario id: ',idcomentario)
      ptoPartida = [idcomentario, pagina] # Punto de partida para volver a comenzar el motor de busqueda en produccion.
      break

    time.sleep(espera) # Tiempo de espera por cada solicitud
    r = requests.get(URIcr+key+'&part='+part+'&fields='+fields+'&parentId='+idcomentario\
                     +'&maxResults='+maxResults+'&pageToken='+pageToken+'&textFormat='+textFormat) # Solicitud
    
    print('solicitud',solicitud,r.reason)

    if (r.reason=='OK'): # Vemos si la solicitud fue exitosa

      d = dict(r.json())
      claves=list(d.keys())
      items = d.get('items')
      Data=pd.json_normalize(items).assign(Solicitud=solicitud)
      Datac=pd.concat([Datac, Data], ignore_index=True)

      if ('nextPageToken' in claves): # Si hay siguiente pagina
        pageToken=d['nextPageToken'] # Definimos el proximo pagetoken
        if ('prevPageToken' in claves):
          prevPageToken=d['prevPageToken']
        else:
          prevPageToken=''

        Metadatac = Metadatac.append({'Solicitud':solicitud,'idComentarioPrincipal':idcomentario,'maxResultados':maxResults\
                                      ,'FechaEjecucion':str(datetime.now()),'Respuesta':r.reason\
                                      ,'prevPageToken':prevPageToken,'nextPageToken':d['nextPageToken']\
                                      ,'Resultadosxpagina':d['pageInfo']['resultsPerPage']}, ignore_index=True)
        print("Siguiente pagina")

      else: # Si no hay siguiente pagina
        nextPageToken=''  # Definimos el nextPageToken como vacio
        if 'prevPageToken' in claves:
          prevPageToken=d['prevPageToken']
        else:
          prevPageToken=''
        
        Metadatac = Metadatac.append({'Solicitud':solicitud,'idComentarioPrincipal':idcomentario,'maxResultados':maxResults\
                                      ,'FechaEjecucion':str(datetime.now()),'Respuesta':r.reason\
                                      ,'prevPageToken':prevPageToken,'nextPageToken':nextPageToken\
                                      ,'Resultadosxpagina':d['pageInfo']['resultsPerPage']}, ignore_index=True)
        print("No hay mas paginas")
        break

    else: # Vemos si la solicitud no fue exitosa
      Faltantes=Faltantes.append({'idComentarioPrincipal': idcomentario,'Solicitud':solicitud}, ignore_index=True)
      print("La solicitud no se pudo realizar para el comentario ", idcomentario)
      break

print("Terminados los comentarios")

In [ ]:
ptoPartida

In [ ]:
indice=ListaIDs.index(ptoPartida[0])
indice

In [ ]:
ListaIDs[indice]

In [ ]:
Datac.info()

In [ ]:
Datac

In [ ]:
Metadatac.info()

In [ ]:
Metadatac

In [ ]:
Faltantes

#### Estructurar y dar formato a Dataframe

In [ ]:
columnas={'snippet.parentId':'idComentarioPrincipal','id':'idComentarioRespuesta'
          ,'snippet.textOriginal':'TextoComentarioRespuesta','snippet.authorDisplayName':'NombreAutorComentarioRespuesta'\
          ,'snippet.authorChannelUrl':'URLAutorCanalComentarioRespuesta','snippet.authorChannelId.value':'idAutorCanalComentarioRespuesta'\
          ,'snippet.canRate':'CalificarComentarioRespuesta','snippet.viewerRating':'CalificacionEspectadorComentarioRespuesta'\
          ,'snippet.likeCount':'LikesComentarioRespuesta','snippet.publishedAt':'FechaPublicacionComentarioRespuesta'\
          ,'snippet.updatedAt':'FechaActualizacionComentarioRespuesta','Solicitud':'SolicitudComentarioRespuesta'}

columnasorden=['idComentarioPrincipal','idComentarioRespuesta','TextoComentarioRespuesta','LikesComentarioRespuesta','FechaPublicacionComentarioRespuesta','FechaActualizacionComentarioRespuesta'\
               ,'NombreAutorComentarioRespuesta','URLAutorCanalComentarioRespuesta','idAutorCanalComentarioRespuesta','CalificacionEspectadorComentarioRespuesta','CalificarComentarioRespuesta','SolicitudComentarioRespuesta']

Datac.rename(columns=columnas,inplace=True)
DATA=Datac.reindex(columns=columnasorden)
DATA

In [ ]:
DATA.info()

#### Extraccion de datos

Se generan 4 nuevas columnas (emoticones, hashtags, URLs y menciones) por cada columna de texto importante. (En este caso 4)

##### Extreaer emoticones

In [ ]:
def emojis(texto):
  if (len(texto) != 0):
    emoji_summary = adv.extract_emoji(texto)
    return emoji_summary['top_emoji']
  else:
    L=[]
    return L

In [ ]:
DATA['emojisComentarioRespuesta']=DATA['TextoComentarioRespuesta'].apply(emojis)

In [ ]:
DATA['emojisComentarioRespuesta']

##### Extraer URLs

In [ ]:
extractor = URLExtract()
def URLs(texto):
  URL=extractor.find_urls(texto)
  return URL

In [ ]:
DATA['URLsComentarioRespuesta']=DATA['TextoComentarioRespuesta'].apply(URLs)

In [ ]:
DATA['URLsComentarioRespuesta']

##### Quitar codigo hexadecimal ( \&quot; , &\#23; , etc) (Revizado)

In [ ]:
def codigohex(texto):
  texto=html.unescape(texto)
  return texto

In [ ]:
DATA['ComentarioRespuestaModificado']=DATA['TextoComentarioRespuesta'].apply(codigohex)

##### Extraer Hastags

In [ ]:
def hastags(texto):
  pattern='#\w+'
  hastag=regexp_tokenize(texto,pattern)
  return hastag

In [ ]:
DATA['hashtagsComentarioRespuesta']=DATA['ComentarioRespuestaModificado'].apply(hastags)

In [ ]:
DATA['hashtagsComentarioRespuesta']

##### Extraer menciones o etiquetas (puede ser necesario alguna modificacion para el caso de correos)

In [ ]:
def mencion(texto):
  pattern='@\w+'
  mencion=regexp_tokenize(texto,pattern)
  return mencion

In [ ]:
DATA['MencionesComentarioRespuesta']=DATA['TextoComentarioRespuesta'].apply(mencion)

In [ ]:
DATA['MencionesComentarioRespuesta']

#### DATA

In [ ]:
DATA.info()

#### Limpiar texto: (Generamos una copia llamada columname_modificado de una columna a partir del cual vamos a realizar la limpieza de datos) OJO: Revisar secuencias de escape para serie tres caines y los 80

##### Quitar codigo hexadecimal ( \&quot; , &\#23; , etc) (Revizado)

In [ ]:
def codigohex(texto):
  texto=html.unescape(texto)
  return texto

In [ ]:
DATA['ComentarioRespuestaModificado']=DATA['TextoComentarioRespuesta'].apply(codigohex)

In [ ]:
DATA['ComentarioRespuestaModificado']

##### Eliminar Unicode (Es aca donde se eliminan las tildes-> Solucionado) NOTA: No se entiende muy bien que realiza este parte del codigo

In [ ]:
def dropunicode(texto):
  texto = unicodedata.normalize('NFD', texto)
  texto=texto.encode('ascii','ignore').decode('utf8')
  return texto

In [ ]:
DATA['ComentarioRespuestaModificado']=DATA['ComentarioRespuestaModificado'].apply(dropunicode)

In [ ]:
DATA['ComentarioRespuestaModificado']

##### Eliminar secuencias de escape

In [ ]:
DATA['TextoComentarioRespuesta'] = DATA['TextoComentarioRespuesta'].apply(str.replace, args=('\r', ' '))
DATA['TextoComentarioRespuesta'] = DATA['TextoComentarioRespuesta'].apply(str.replace, args=('\n', ' '))
DATA['TextoComentarioRespuesta'] = DATA['TextoComentarioRespuesta'].apply(str.replace, args=('\v', ' '))
DATA['TextoComentarioRespuesta'] = DATA['TextoComentarioRespuesta'].apply(str.replace, args=('\t', ' '))
DATA['TextoComentarioRespuesta'] = DATA['TextoComentarioRespuesta'].apply(str.replace, args=('\f', ' '))
DATA['TextoComentarioRespuesta'] = DATA['TextoComentarioRespuesta'].apply(str.replace, args=('\b', ' '))
#DATA['TextoComentarioRespuesta'] = DATA['TextoComentarioRespuesta'].apply(str.replace, args=('3', 'tres')) # solo para tres caines
#DATA['TextoComentarioRespuesta'] = DATA['TextoComentarioRespuesta'].apply(str.replace, args=('80', 'ochenta')) # solo para los ochenta
DATA['ComentarioRespuestaModificado'] = DATA['ComentarioRespuestaModificado'].apply(str.replace, args=('\r', ' '))
DATA['ComentarioRespuestaModificado'] = DATA['ComentarioRespuestaModificado'].apply(str.replace, args=('\n', ' '))
DATA['ComentarioRespuestaModificado'] = DATA['ComentarioRespuestaModificado'].apply(str.replace, args=('\v', ' '))
DATA['ComentarioRespuestaModificado'] = DATA['ComentarioRespuestaModificado'].apply(str.replace, args=('\t', ' '))
DATA['ComentarioRespuestaModificado'] = DATA['ComentarioRespuestaModificado'].apply(str.replace, args=('\f', ' '))
DATA['ComentarioRespuestaModificado'] = DATA['ComentarioRespuestaModificado'].apply(str.replace, args=('\b', ' '))
#DATA['ComentarioRespuestaModificado'] = DATA['ComentarioRespuestaModificado'].apply(str.replace, args=('3', 'tres')) # solo para tres caines
#DATA['ComentarioRespuestaModificado'] = DATA['ComentarioRespuestaModificado'].apply(str.replace, args=('80', 'ochenta')) # solo para los ochenta

##### Eliminar URLs

In [ ]:
def dropURLs(texto):
  text = re.sub('https?://\S+|www\.\S+', ' ', texto)
  return text
def dropURL(texto):
  text = re.sub('http?://\S+|www\.\S+', ' ', texto)
  return text

In [ ]:
DATA['ComentarioRespuestaModificado']=DATA['ComentarioRespuestaModificado'].apply(dropURLs)
DATA['ComentarioRespuestaModificado']=DATA['ComentarioRespuestaModificado'].apply(dropURL)

In [ ]:
DATA['ComentarioRespuestaModificado']

##### Eliminar Emoticones

In [ ]:
from cleantext import clean

In [ ]:
from dataclasses import replace
def dropemoticons(texto):
  texto=clean(texto, no_emoji=True,lower=False)
  return texto

In [ ]:
DATA['ComentarioRespuestaModificado']=DATA['ComentarioRespuestaModificado'].apply(dropemoticons)

In [ ]:
DATA['ComentarioRespuestaModificado']

##### Eliminar emails

In [ ]:
def dropemails(texto):
  texto=clean(texto, no_emails=True, lower=False)
  return texto

In [ ]:
DATA['ComentarioRespuestaModificado']=DATA['ComentarioRespuestaModificado'].apply(dropemails)

In [ ]:
DATA['ComentarioRespuestaModificado']

##### Eliminar Menciones

In [ ]:
def dropmenciones(texto):
  texto = re.sub("@\S+", " ", texto)
  return texto

In [ ]:
DATA['ComentarioRespuestaModificado']=DATA['ComentarioRespuestaModificado'].apply(dropmenciones)

In [ ]:
DATA['ComentarioRespuestaModificado']

##### Quitar caracteres repetidos


In [ ]:
def droppalabrasrepetidas(texto):
  texto = re.sub(r'(.)\1{3,}',r'\1', texto)
  return texto

In [ ]:
DATA['ComentarioRespuestaModificado']=DATA['ComentarioRespuestaModificado'].apply(droppalabrasrepetidas)

In [ ]:
DATA['ComentarioRespuestaModificado']

##### Eliminar Numeros

In [ ]:
def dropnumeros(texto):
  pattern = r'[0-9]'
  texto = re.sub(pattern, '', texto)
  return texto

In [ ]:
DATA['ComentarioRespuestaModificado']=DATA['ComentarioRespuestaModificado'].apply(dropnumeros)

In [ ]:
DATA['ComentarioRespuestaModificado']

##### Eliminar \# de Hashtags

In [ ]:
def dropnumeral(texto):
  texto = re.sub(r'#', ' ', texto)
  return texto

In [ ]:
DATA['ComentarioRespuestaModificado']=DATA['ComentarioRespuestaModificado'].apply(dropnumeral)

In [ ]:
DATA['ComentarioRespuestaModificado']

##### Dividir palabras adjuntas (CamelCase)

In [ ]:
def splitpalabras(texto):
  texto = " ".join([s for s in re.split("([A-Z][a-z]+[^A-Z]*)",texto) if s])
  return texto

In [ ]:
DATA['ComentarioRespuestaModificado']=DATA['ComentarioRespuestaModificado'].apply(splitpalabras)

In [ ]:
DATA['ComentarioRespuestaModificado']

##### Eliminar signos de puntuacion

In [ ]:
punct = set(string.punctuation) 
punct 

In [ ]:
def puntuacion(texto):
  sinpuntuacion="".join([ch for ch in texto if ch not in punct])
  return sinpuntuacion

In [ ]:
DATA['ComentarioRespuestaModificado']=DATA['ComentarioRespuestaModificado'].apply(puntuacion)

In [ ]:
DATA['ComentarioRespuestaModificado']

##### Eliminar espacios

In [ ]:
def dropespacios(texto):
  texto = re.sub('\s{2,}', " ", texto)
  return texto

In [ ]:
DATA['ComentarioRespuestaModificado']=DATA['ComentarioRespuestaModificado'].apply(dropespacios)

In [ ]:
DATA['ComentarioRespuestaModificado']

##### Eliminar tokens cortos

In [ ]:
def droppalabrascortas(texto):
  tokens = texto.split()
  clean_tokens = [t for t in tokens if len(t) > 1]
  clean_text = " ".join(clean_tokens)
  return clean_text

In [ ]:
DATA['ComentarioRespuestaModificado']=DATA['ComentarioRespuestaModificado'].apply(droppalabrascortas)

In [ ]:
DATA['ComentarioRespuestaModificado']

##### Normalizar texto (Quitar mayusculas)

In [ ]:
def normatext(texto):
  textonormalizado=texto.lower()
  return textonormalizado

In [ ]:
DATA['ComentarioRespuestaModificado']=DATA['ComentarioRespuestaModificado'].apply(normatext)

In [ ]:
DATA['ComentarioRespuestaModificado']

##### Cambios particulares para cada serie

In [ ]:
def cambiosEPM(texto):
  texto1=re.sub("elpatrondelmal", "el patron del mal", texto)
  texto2=re.sub("patrondelmal", "patron del mal", texto1)
  texto3=re.sub("patrondel", "patron del", texto2)
  texto4=re.sub("pabloescobar", "pablo escobar", texto3)
  texto5=re.sub("murtes", "muertes", texto4)
  texto6=re.sub("murte", "muerte", texto5)
  texto7=re.sub("muerts", "muertes", texto6)
  texto8=re.sub("patron", " patron ", texto7)
  texto9=re.sub("escobar", " escobar ", texto8)
  texto=re.sub("pablo", " pablo ", texto9)
  return texto

In [ ]:
DATA['ComentarioRespuestaModificado']=DATA['ComentarioRespuestaModificado'].apply(cambiosEPM)

In [ ]:
DATA['ComentarioRespuestaModificado']

##### Eliminando stopwords

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk import word_tokenize
nltk.download('stopwords')
nltk.download('punkt')

In [ ]:
swsp=nltk.corpus.stopwords.words('spanish')

In [ ]:
swpo=nltk.corpus.stopwords.words('portuguese')

In [ ]:
swen=nltk.corpus.stopwords.words('english')

In [ ]:
def dropStopWordsSpanish(texto):
  if (str(texto) != 'nan'):
    tokens=word_tokenize(texto)
    tokensssw=[]
    for w in tokens:
      if w not in swpo:
        tokensssw.append(w)
    texto=" ".join(tokensssw)
  return texto

In [ ]:
def dropStopWordsPortugues(texto):
  if (str(texto) != 'nan'):
    tokens=word_tokenize(texto)
    tokensssw=[]
    for w in tokens:
      if w not in swsp:
        tokensssw.append(w)
    texto=" ".join(tokensssw)
  return texto

In [ ]:
def dropStopWordsEnglish(texto):
  if (str(texto) != 'nan'):
    tokens=word_tokenize(texto)
    tokensssw=[]
    for w in tokens:
      if w not in swen:
        tokensssw.append(w)
    texto=" ".join(tokensssw)
  return texto

In [ ]:
DATA['ComentarioRespuestaModificado']=DATA['ComentarioRespuestaModificado'].apply(dropStopWordsSpanish)
DATA['ComentarioRespuestaModificado']=DATA['ComentarioRespuestaModificado'].apply(dropStopWordsPortugues)
DATA['ComentarioRespuestaModificado']=DATA['ComentarioRespuestaModificado'].apply(dropStopWordsEnglish)

In [ ]:
DATA['ComentarioRespuestaModificado']

##### Volvemos a eliminar exceso de espacios

In [ ]:
DATA['ComentarioRespuestaModificado']=DATA['ComentarioRespuestaModificado'].apply(dropespacios)

In [ ]:
DATA['ComentarioRespuestaModificado']

##### Volvemos a eliminar tokens cortos

In [ ]:
DATA['ComentarioRespuestaModificado']=DATA['ComentarioRespuestaModificado'].apply(droppalabrascortas)

In [ ]:
DATA['ComentarioRespuestaModificado']

#### Estructurar y dar formato

In [ ]:
DATA.info()

In [ ]:
columnasorden=['idComentarioPrincipal','idComentarioRespuesta','TextoComentarioRespuesta','ComentarioRespuestaModificado','emojisComentarioRespuesta','URLsComentarioRespuesta','hashtagsComentarioRespuesta','MencionesComentarioRespuesta'\
               ,'LikesComentarioRespuesta','FechaPublicacionComentarioRespuesta','FechaActualizacionComentarioRespuesta'\
               ,'NombreAutorComentarioRespuesta','URLAutorCanalComentarioRespuesta','idAutorCanalComentarioRespuesta','CalificacionEspectadorComentarioRespuesta','CalificarComentarioRespuesta','SolicitudComentarioRespuesta']

DATA=DATA.reindex(columns=columnasorden)
DATA

#### Exportar Dataframe a CSV

In [ ]:
Metadatac.to_csv(PATH_TO_DIR_0+'/'+file_nameL+'.csv', index=False)
Faltantes.to_csv(PATH_TO_DIR_0+'/'+'faltantes.csv', index=False)
DATA.to_csv(PATH_TO_DIR_0+'/'+'RESULTS-'+file_nameL+'.csv', index=False)

## Concatenando bases de datos de respuestas comentarios

In [ ]:
# ================== OJO modificar para cada serie ==================

RespuestasComentarios1 = pd.read_csv ('/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/YoutubeResults/RespuestasComentarios/\
Narcos/2022-09-13/RESULTS-RESPUESTASCOMENTARIOS2022-09-13 17:52:58.638476.csv')

RespuestasComentarios2 = pd.read_csv ('/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/YoutubeResults/RespuestasComentarios/\
Narcos/2022-09-14/RESULTS-RESPUESTASCOMENTARIOS2022-09-14 14:58:51.668421.csv')

# ================== OJO modificar para cada serie ==================


In [ ]:
RespuestasComentarios1

In [ ]:
RespuestasComentarios2

In [ ]:
DATA=pd.concat([RespuestasComentarios1,RespuestasComentarios2])

In [ ]:
DATA

In [ ]:
DATAclear=DATA.drop_duplicates(subset=['idComentarioRespuesta'])
DATAclear.info()

In [ ]:
today = str(date.today())

# ======================= OJO Cambiar PATH para cada Serie =======================

PATH_TO_SAVE_SEARCHES = '/content/drive/MyDrive/AI-GUMELAB/CognASSISTANT-1/SEARCHES/YoutubeResults/RespuestasComentarios/Narcos/'

# ======================= OJO Cambiar PATH para cada Serie =======================

PATH_TO_DIR_0 = PATH_TO_SAVE_SEARCHES + today+'RC1(2022-09-13)'+'RC2(2022-09-14)'
if os.path.exists(PATH_TO_DIR_0) == False:
  os.mkdir(PATH_TO_DIR_0)
file_nameL = 'RESPUESTASCOMENTARIOS'+str(datetime.now())

In [ ]:
DATAclear.to_csv(PATH_TO_DIR_0+'/'+'RESULTS-'+file_nameL+'.csv', index=False)